# A Lexicon of Black Institutional Recognition Practices
## HBCU Digital Library Trust Graph Artifact

This notebook is a minimal methodological artifact for a proof-of-concept graph model. It uses HBCU Digital Library Trust documentation to derive a source-grounded relational lexicon for representing forms of labor, care, stewardship, access, metadata, rights, recognition, return, and preservation that conventional citation or bibliography maps often miss.

**Research question:** How can graph modeling be used to represent the forms of stewardship, preservation, recognition, metadata labor, institutional care, and access that sustain HBCU knowledge production but are often invisible within conventional citation and bibliography maps?


## 1. Corpus manifest

The source manifest records the bounded corpus for the proof of concept. The corpus is not meant to represent the entire HBCU Digital Collection. It is a small, auditable set of project documents that describe the Trust's mission, workflows, metadata practices, rights structures, and access systems.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
sources = pd.read_csv(DATA_DIR / "sources.csv")
sources[["source_id", "title", "source_type", "role_in_project"]]

## 2. Extracted page-level text

In the full pipeline, page-level text is extracted from PDF/web-printout files using Python and PyMuPDF. The page-level table keeps source identifiers and page numbers attached to extracted text so that later edges can be audited against evidence.

For a submitted notebook, the extracted text is loaded from `data/extracted_pages.csv`. This keeps the artifact runnable even when raw PDFs are not distributed with the notebook.

In [ ]:
extracted_pages = pd.read_csv(DATA_DIR / "extracted_pages.csv")
extracted_pages[["source_id", "page", "text"]].head()

## 3. Candidate relationship extraction

This step is intentionally lightweight. It does not claim to automatically discover relationships. It searches for recurring terms associated with preservation, access, metadata, rights, digitization, collaboration, and memory. The output is a set of evidence windows for qualitative review.

This is why I describe the method as **script-assisted qualitative graph construction** rather than automated extraction.

In [ ]:
candidate_edges = pd.read_csv(DATA_DIR / "candidate_edges.csv")
candidate_edges.head(12)

## 4. Relational lexicon

The lexicon preserves source-grounded relation verbs. These verbs are not generic predicates imposed after the fact. Some come from workflow language, such as `SCANS`, `QUALITY_CHECKS`, `UPLOADS_TO`, and `RETURNS_TO`. Some come from mission or value language, such as `BUILDS_CAPACITY_WITH`, `PRESERVES`, `SUSTAINS`, and `ACKNOWLEDGES_AND_CELEBRATES`. Others translate metadata, rights, and research access practices into graph form, such as `DESCRIBE`, `LINK_COMMONALITY_ACROSS`, and `RETAINS_OWNERSHIP_OF`.

This is why the relation verbs are treated as part of the scholarly argument rather than as a purely technical schema.


In [ ]:
lexicon = pd.read_csv(DATA_DIR / "lexicon.csv")
lexicon[["term", "category", "relation_grounding", "working_definition", "paper_function"]].head(20)


## 5. Normalized nodes and evidence-bearing edges

The final graph uses two Neo4j-ready CSV files: `nodes.csv` and `edges.csv`. The tables are intentionally small. The goal is not comprehensive coverage. The goal is to test whether graph modeling can represent forms of stewardship and institutional care that are usually absent from citation maps.

The enhanced edge table preserves the source-derived `relationship` verb and adds `relation_family`, `relation_grounding`, `relation_definition`, and `paper_function` so that the graph can be read as an interpretive artifact rather than only as network data.


In [ ]:
nodes = pd.read_csv(DATA_DIR / "nodes.csv")
edges = pd.read_csv(DATA_DIR / "edges.csv")

print(f"Nodes: {len(nodes)}")
print(f"Edges: {len(edges)}")
nodes.head(10)

In [ ]:
edges[["source_name", "relationship", "target_name", "relation_family", "relation_grounding", "evidence_source_id", "evidence_page"]].head(15)


## 5.1 Schema validation

This check confirms that every edge connects existing nodes, uses a relation included in the expanded lexicon, and retains a valid evidence source.


In [ ]:
node_ids = set(nodes['node_id'])
source_ids = set(sources['source_id'])
relation_terms = set(lexicon['term'])
validation_errors = []
for _, e in edges.iterrows():
    if e['source_node_id'] not in node_ids:
        validation_errors.append((e['edge_id'], 'missing source node'))
    if e['target_node_id'] not in node_ids:
        validation_errors.append((e['edge_id'], 'missing target node'))
    if e['relationship'] not in relation_terms:
        validation_errors.append((e['edge_id'], 'relationship not in lexicon'))
    if e['evidence_source_id'] not in source_ids:
        validation_errors.append((e['edge_id'], 'missing evidence source'))

print('Validation errors:', len(validation_errors))
pd.DataFrame(validation_errors, columns=['edge_id', 'issue']).head()


## 6. Quick graph preview with NetworkX

This preview is not the final argument. It is a quick check on whether the extracted actors and relationships form a plausible model before importing to Neo4j.

In [ ]:
from IPython.display import Image, display

display(Image(filename="outputs/graph_preview.png"))

## 7. Neo4j import sketch

Neo4j imports the final CSV files rather than Excel workbooks. CSV is transparent, lightweight, and compatible with `LOAD CSV`. The `scripts/neo4j_import.cypher` file contains a minimal import script.

In [ ]:
print(Path("scripts/neo4j_import.cypher").read_text()[:1200])

## 8. Evaluation questions

The graph should be evaluated qualitatively rather than only through graph metrics.

1. What relationships become visible when stewardship, preservation, description, access, rights, and return are modeled alongside influence?
2. Which actors become visible when the graph includes librarians, archivists, metadata workers, digitization staff, platforms, rights statements, and preservation files?
3. What remains difficult to represent even with expanded relationship verbs?
4. How did corpus boundaries and node/edge definitions shape the model?
5. What would change if this lexicon were applied to a future HBCU bibliography or citation map?

## 9. Methodological limits

This graph does not represent the full HBCU Digital Library Trust, the entire HBCU Digital Collection, or all HBCU knowledge production. It is a proof of concept. Its relations are derived from a bounded set of public and contributor-facing documents. The model makes some forms of work visible, but it also necessarily flattens affective, historical, ethical, and institutional meanings that cannot be fully captured as nodes and edges.

The point is not to automate Black knowledge production. The point is to show that citation maps can be expanded when we learn relational categories from Black archival and institutional infrastructures themselves.